[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/autogen/autogen_agents_evaluator_optimizer.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# AutoGen 智能体：评估与优化

本教程使用 AutoGen [群聊](https://microsoft.github.io/autogen/dev//user-guide/core-user-guide/design-patterns/group-chat.html) 探索**评估与优化智能体模式**，并展示如何通过 Phoenix 追踪过程。

这一模式采用循环：生成智能体创建初始文本或代码，评估智能体按标准给出评判与反馈。反馈指导生成器连续修改，通过迭代提升复杂任务的质量，以更多交互换取更完善、准确的最终结果。

AutoGen 的 `GroupChat` 架构可管理生成与评估智能体的对话轮次，适合实现这一模式。`GroupChatManager` 协调对话，让智能体交换不断变化的输出与反馈。

Phoenix 追踪可以完整展示每轮改进，跟踪反馈、修改，以及朝目标质量标准推进的进度，便于调试和分析。

完成本教程后，你将学会：

- 配置具有明确角色和标准的 AutoGen 生成与评估智能体。
- 在 AutoGen GroupChat 中实现评估与优化模式。
- 管理智能体之间的迭代改进循环。
- 使用 Phoenix 追踪并可视化反馈与修改过程。

⚠️ 本教程需要 OpenAI 密钥。

## 设置密钥与依赖

In [ ]:
!pip install -qqq arize-phoenix arize-phoenix-otel openinference-instrumentation-openai

In [ ]:
!pip install -qqq pyautogen==0.9 autogen-agentchat~=0.2

In [ ]:
import os
from getpass import getpass

import autogen

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥： ")

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

## 配置追踪

In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name="autogen-agents",
    auto_instrument=True,
)

## 评估与优化示例任务：代码生成与评估

**评估与优化**智能体模式通过循环，让一个 AI 智能体生成代码等输出，由另一个智能体按具体标准评估并推动改进。

本例中，`Code_Generator` 根据需求编写 Python 代码，专用 `Code_Reviewer` 检查正确性、风格与文档，并提供针对性反馈。通过 AutoGen `GroupChat` 中反复生成与评审，展示如何产出符合标准、经过评审的高质量代码。

![流程图](https://storage.googleapis.com/arize-phoenix-assets/assets/images/autogen_evaluate_optimizer_diagram.png)

## 定义智能体

`llm_config` 指定各 `AssistantAgent` 使用的配置。
代码生成和评估使用不同模型。

In [ ]:
config_list_gen = {
    "model": "gpt-4o",
    "api_key": os.environ["OPENAI_API_KEY"],
}

config_list_eval = {
    "model": "gpt-4.1-mini",
    "api_key": os.environ["OPENAI_API_KEY"],
}
llm_config_gen = {
    "config_list": config_list_gen,
    "temperature": 0.5,
}  # 温度参数控制生成的灵活性
llm_config_eval = {"config_list": config_list_eval, "temperature": 0.5}

本节初始化两个用于代码生成与评审迭代的 AutoGen `AssisstantAgents`。

`Code_Generator` 根据指令与反馈编写和修改 Python 代码；`Code_Reviewer` 按正确性、风格、可读性等详细标准评估代码，给出具体反馈，或精确终止信号：`"TERMINATE!"`。

In [ ]:
# 代码生成智能体
coder = autogen.AssistantAgent(
    name="Code_Generator",
    system_message="""你是 Python 编程专家。你的目标是根据用户请求编写正确、高效、整洁的 Python 代码。
    你会收到代码需求，或代码评审者提供的反馈请求。
    编写代码时：
    - 始终将完整 Python 代码块放在 ```python ... ``` 标记内。
    - 确保代码直接解决请求。
    - 包含必要导入。
    - 在适当位置添加文档字符串和注释。

    收到代码评审者反馈时：
    - 仔细分析每条反馈。
    - 准确落实建议，重写代码块。
    - 仅输出完整修改后的 Python 代码块，不添加文本或解释。""",
    llm_config=llm_config_gen,
)

# 代码评估智能体
reviewer = autogen.AssistantAgent(
    name="Code_Reviewer",
    system_message="""
    你是专注 Python 的代码评审专家。请根据原始请求及以下标准评估 Coder 智能体编写的代码：
    1. 正确性：逻辑是否正确，能否满足请求要求？
    2. 规范性：是否遵循标准 Python 风格，包括命名、缩进、行长？
    3. 文档与注释：文档字符串是否清楚说明函数或类的用途、参数与返回值？是否有效使用必要注释？
    4. 可读性与可维护性：代码是否易懂？变量名是否有意义？逻辑是否直接？
    5. 效率：典型用例下是否有明显的重大性能问题或极低效模式？
    6. 错误处理：是否考虑相关潜在错误与边界情况？

    全面评审代码。提供具体、建设性、带编号的反馈，尽可能引用行号，重点提出满足标准所需的可操作改进。
    如果代码符合全部标准且无需进一步修改，仅回复以下精确短语：TERMINATE!
    使用终止短语批准代码时，不要提供对话文本或总结。""",
    llm_config=llm_config_eval,
)

接下来定义 `check_reviewer_approval` 函数，专门检测消息是否仅包含 `"TERMINATE!"`。

随后初始化 AutoGen `UserProxyAgent` 以扮演人类用户，并将 `check_reviewer_approval` 作为该智能体的终止消息检查函数。

In [ ]:
def check_reviewer_approval(message_dict):
    content = message_dict.get("content")
    if isinstance(content, str) and content.strip() == "TERMINATE!":
        return True
    return False


user_proxy = autogen.UserProxyAgent(
    name="User_Proxy",
    system_message="提供初始代码需求并接收最终结果的人类用户，同时执行终止检查。",
    human_input_mode="NEVER",
    is_termination_msg=check_reviewer_approval,
    code_execution_config=False,
)

最后初始化智能体框架：先定义包含用户代理、代码生成器与评审器的 AutoGen `GroupChat`，最多交互 10 轮。

然后创建 `GroupChatManager` 编排群内对话，为其指定 LLM 配置，并传入 `check_reviewer_approval` 函数以识别终止条件。`GroupChatManager` 根据已有上下文决定下一步调用哪个智能体。

In [ ]:
groupchat = autogen.GroupChat(agents=[user_proxy, coder, reviewer], messages=[], max_round=10)

manager = autogen.GroupChatManager(
    groupchat=groupchat, llm_config=llm_config_gen, is_termination_msg=check_reviewer_approval
)

## 运行智能体

现在可以使用示例 `coding_task` 运行智能体。

In [ ]:
coding_task = """
请编写名为 `calculate_fibonacci(n)` 的 Python 函数，计算第 n 个斐波那契数。

要求：
1. 函数接收非负整数 `n`。
2. 返回第 n 个斐波那契数。
3. 提供清晰文档字符串，解释函数、参数和返回值。
4. 正确处理基础情况。
5. 使用迭代方法提高效率。
"""

print("--- 开始代码生成任务 ---")
print(f"请求： {coding_task}\n")

tracer = tracer_provider.get_tracer(__name__)
with tracer.start_as_current_span(
    "CodeGenEval",
    openinference_span_kind="agent",
) as agent_span:
    # 启动对话
    user_proxy.initiate_chat(
        manager,
        message=coding_task,
    )

print("--- 代码生成任务结束 ---")

## 在 Phoenix 中查看结果

Phoenix 追踪通过可视化 `GroupChat` 中 `Code_Generator` 和 `Code_Reviewer` 之间的迭代循环，展示评估与优化工作流。

追踪中可以查看完整改进过程，检查每一步发送给智能体的具体提示词，以及输出在终止决策前如何逐渐变化和完善。

运行下方单元格查看完整追踪结果。

In [ ]:
from IPython.display import HTML

HTML("""
<video width="800" height="600" controls>
  <source src="https://storage.googleapis.com/arize-phoenix-assets/assets/videos/autogen_eval_optimizer_results.mp4" type="video/mp4">
  浏览器不支持 video 标签。
</video>
""")